In [ ]:
text = """
Predetermination Evidence During Pending Appeal
On 11 April 2024, while my appeal against dismissal was formally pending and before any appeal hearing
had taken place or outcome reached, a former colleague contacted me via LinkedIn to express sympathy for
what had “happened” to me. This communication was unexpected and concerning, as the dismissal decision
was expressly under appeal and had not been concluded or upheld at that stage.
Following this initial contact, the discussion moved to WhatsApp. During that exchange, my former
colleague informed me, in substance, that “we were told that you manipulated the system to selectively
choose clients”. I was told that this allegation had been communicated to the team by my Team Leader at the
time, Shota Wada, during a team meeting in which the reasons relied upon for my dismissal were disclosed.
The allegation was presented as an established conclusion, rather than as a contested or unresolved matter
subject to appeal. A screenshot of this exchange is available.
At the time this allegation was being communicated internally, the appeal process was ongoing and
unresolved. The circulation of such a serious allegation prior to the conclusion of the appeal demonstrates
that the dismissal decision was being treated internally as final, rather than as subject to genuine
reconsideration. This is incompatible with the purpose of an appeal, which is to provide an independent and
open-minded review of the original decision.
I was subsequently informed that the source of this disclosure was management, specifically my Team
Leader at the relevant time. This confirms that the dissemination of the allegation did not arise from informal
speculation or misunderstanding among colleagues, but from within the management structure itself. The
communication of contested disciplinary allegations to an external third party, with no role in the
disciplinary or appeal process, constitutes an unauthorised disclosure of confidential and personal
information.
The disclosure of the reason for my dismissal had material and adverse consequences. In the period
following my dismissal and while my appeal was pending, I applied for multiple roles in London within the
same sector. In a number of cases, recruitment processes that had progressed to advanced stages abruptly
ceased without explanation, despite prior positive engagement. I reasonably inferred that the internal
circulation of the allegation, and the manner in which it was presented as an established finding, had
damaged my professional reputation and prospects. As a result, I was unable to secure comparable
employment in London and was compelled to relocate in order to obtain work.
In parallel, I note that I had joined the Fishbowl application well before using my corporate email address.
Fishbowl is an anonymous platform, and my access to it pre-dated the events giving rise to my dismissal.
Within an internal Bloomberg-specific forum on that platform, multiple employees were openly discussing
practices relating to performance management, including the use of warnings, performance improvement
plans, and dismissals as indirect headcount-reduction mechanisms. Those discussions were ongoing at the
time of my dismissal and appeal and provide contemporaneous context as to the broader environment in
which disciplinary decisions were being taken. I do not rely on the accuracy of any individual post; rather, I
rely on the existence of such discussions as background context.
No steps were taken during the appeal process to address or investigate the internal dissemination of the
allegation, nor to correct the presentation of the alleged misconduct as an established fact while the appeal
remained live. This failure further undermined the integrity of the appeal process.
The seriousness of this conduct is twofold. First, it evidences that the appeal outcome had been
predetermined, as management were communicating the alleged misconduct as fact while the appeal was still
ongoing. Second, it represents a fundamental breach of confidentiality and procedural integrity, undermining
trust in the fairness, independence, and legitimacy of the appeal process as a whole.
Taken together with the matters set out earlier in this statement — including the lack of prior warning or
support, the failure to engage with workload and system-instability concerns, the reliance on isolated and
selective examples, and the comparator evidence demonstrating materially different treatment — I concluded
that the Respondent did not hold a genuine belief in misconduct reached through a fair and open-minded
process. Rather, the evidence supports an inference that the disciplinary process was directed toward
individuals with limited internal support or advocacy, including those from minority backgrounds, as
addressed in the relevant sections above. This inference is drawn from the cumulative procedural failures and
inconsistencies identified, not from any single matter viewed in isolation.

"""


In [ ]:
import subprocess
import json
import re
from typing import Any, Dict

def _run_ollama(prompt: str, model: str) -> str:
    r = subprocess.run(
        ["ollama", "run", model],
        input=prompt,
        text=True,
        capture_output=True
    )
    if r.returncode != 0:
        raise RuntimeError(f"Ollama error:\n{r.stderr}")
    return (r.stdout or "").strip()

def _extract_json_object(s: str) -> Dict[str, Any]:
    """
    Extract the first top-level JSON object from a string.
    Handles cases where the model prints extra text before/after JSON.
    """
    if not s.strip():
        raise ValueError("Model returned empty output.")

    # Fast path: it's already valid JSON
    try:
        obj = json.loads(s)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

    # Try to locate the first {...} block (including newlines)
    start = s.find("{")
    end = s.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("Could not find a JSON object in the model output.")

    candidate = s[start:end+1].strip()

    # Sometimes there are trailing code fences or garbage — strip common wrappers
    candidate = re.sub(r"^```(json)?\s*", "", candidate, flags=re.IGNORECASE)
    candidate = re.sub(r"\s*```$", "", candidate)

    obj = json.loads(candidate)
    if not isinstance(obj, dict):
        raise ValueError("Extracted JSON is not an object/dict.")
    return obj


Y_PROMPT_V2 = """
You are building a STRICT evaluation rubric called Y to classify whether a short case description
(e.g., 'reasoning_for_index' or 'summary') is useful for a fact pattern X.

OUTPUT:
Return VALID JSON ONLY (no markdown, no code fences, no commentary).
The output must start with { and end with }.

SCHEMA (must match exactly):

{
  "version": "Y_v2",
  "x_tests": {
    "X1": {"name": "...", "definition": "...", "positive_indicators": ["..."], "excludes": ["..."]},
    "X2": {"name": "...", "definition": "...", "positive_indicators": ["..."], "excludes": ["..."]},
    "X3": {"name": "...", "definition": "...", "positive_indicators": ["..."], "excludes": ["..."]},
    "X4": {"name": "...", "definition": "...", "positive_indicators": ["..."], "excludes": ["..."]},
    "X5": {"name": "...", "definition": "...", "positive_indicators": ["..."], "excludes": ["..."]}
  },
  "classes": {
    "DIRECT_X": {"definition": "...", "min_support": 1, "evidence_required": true},
    "CONTRASTIVE": {"definition": "...", "evidence_required": true},
    "REMEDY": {"definition": "...", "evidence_required": true},
    "IRRELEVANT": {"definition": "..."}
  },
  "hard_rules": [
    "rule 1",
    "rule 2",
    "rule 3",
    "rule 4"
  ]
}

INSTRUCTIONS FOR X_TESTS (X1..X5):
- X1..X5 must be ATOMIC and map cleanly to distinct parts of X:
  X1 = appeal treated as final / appeal integrity undermined / predetermination signals
  X2 = dissemination of allegations as fact to colleagues / internally while appeal live
  X3 = disclosure to external third parties / unauthorised disclosure / confidentiality breach
  X4 = management as the source (team leader/management communicated it) rather than gossip
  X5 = adverse reputational/prospects impact flowing from disclosure/predetermination
- Do NOT mix multiple atoms inside one test.
- Keep indicators phrased so they can be detected in short summaries.

CLASS DEFINITIONS:
- DIRECT_X: supports at least one of X1..X4 (misconduct/behaviour undermining appeal integrity or confidentiality).
  X5 alone is not enough for DIRECT_X.
- CONTRASTIVE: describes proper appeal/open-minded review/independent reconsideration (useful comparison), but no misconduct matching X1..X4.
- REMEDY: primarily about compensation/mitigation/Polkey/Johnson v Unisys/Triggs/timelines/causation of loss and NOT about X1..X4.
- IRRELEVANT: none of the above.

HARD RULES (must be enforceable by code):
- DIRECT_X requires supports_X contains at least one of [X1,X2,X3,X4] AND evidence_required=true.
- If only remedy topics appear (compensation/mitigation/timeline) and none of X1..X4 -> REMEDY.
- CONTRASTIVE must mention proper appeal/open-minded/independent review language and exclude misconduct.
- If evidence is missing or vague, prefer IRRELEVANT over DIRECT_X.

Return JSON ONLY.
"""
def generate_Y_v2(X_text: str, model: str = "mistral-small3.2:latest", debug: bool = True):
    prompt = Y_PROMPT_V2 + "\n\nX:\n" + X_text.strip()
    raw = _run_ollama(prompt, model)

    if debug:
        print("---- RAW MODEL OUTPUT (first 1500 chars) ----")
        print(raw[:1500])
        print("---- END RAW ----")

    Y = _extract_json_object(raw)

    # sanity checks
    for k in ["version", "x_tests", "classes", "hard_rules"]:
        if k not in Y:
            raise ValueError(f"Y missing key: {k}")
    if "Y_v2" not in str(Y.get("version","")):
        # not fatal, but helpful
        print("Warning: version is not 'Y_v2'")

    return Y

Y2 = generate_Y_v2(text, model="mistral-small3.2:latest", debug=False)
print(json.dumps(Y2, indent=2))

with open("Y_v2.json", "w") as f:
    json.dump(Y2, f, indent=2)


In [ ]:
import os
import json
import hashlib
from pathlib import Path
from typing import Dict, Any, Iterator, Tuple, Optional, List

import requests

# =========================
# CONFIG
# =========================
BASE_DIR = Path("/home/hello/Projects/FATE_Dr_WB/Dr_WBerious/output/legal-corpus/normalized")

INPUT_FILES = [
    BASE_DIR / "judgments_claimant_fav_faiss_chunk_with_reasoning.jsonl",
    BASE_DIR / "judgments_respondent_fav_faiss_chunk_with_reasoning.jsonl",
    BASE_DIR / "judgments_unknown_with_reasoning_and_factors.jsonl",
]

OUT_DIR = Path("/home/hello/Projects/Statements/output")  # keep it simple; change if you want
Y_JSON_PATH = OUT_DIR / "Y_v2.json"                 # frozen Y
PASS1_OUT   = OUT_DIR / "pass1_results.jsonl"       # streaming output (resume-safe)

MODEL_NAME = "mistral-small3.2:latest"              # <- set your local model here
OLLAMA_URL = "http://localhost:11434/api/generate"  # default Ollama

# If you want to cap scanning during testing:
DEBUG_MAX = None  # e.g. 25


# =========================
# Y_v2 (FROZEN) - paste yours here OR load from disk after saving
# =========================
Y_v2 = {
  "version": "Y_v2",
  "x_tests": {
    "X1": {
      "name": "Appeal treated as final / Appeal integrity undermined / Predetermination signals",
      "definition": "Evidence that the appeal process was compromised by treating the dismissal decision as final or predetermined.",
      "positive_indicators": [
        "appeal outcome had been predetermined",
        "management were communicating the alleged misconduct as fact while the appeal was still ongoing",
        "dismissal decision was being treated internally as final"
      ],
      "excludes": []
    },
    "X2": {
      "name": "Dissemination of allegations as fact to colleagues / Internally while appeal live",
      "definition": "Allegations related to the dismissal were shared internally as established facts during the appeal process.",
      "positive_indicators": [
        "the allegation was presented as an established conclusion",
        "the dismissal decision was being treated internally as final",
        "the circulation of such a serious allegation prior to the conclusion of the appeal"
      ],
      "excludes": []
    },
    "X3": {
      "name": "Disclosure to external third parties / Unauthorised disclosure / Confidentiality breach",
      "definition": "Unauthorized disclosure of confidential information to external parties.",
      "positive_indicators": [
        "communication of contested disciplinary allegations to an external third party",
        "unauthorised disclosure of confidential and personal information"
      ],
      "excludes": []
    },
    "X4": {
      "name": "Management as the source (team leader/management communicated it) rather than gossip",
      "definition": "The source of the disclosure is identified as management rather than informal speculation.",
      "positive_indicators": [
        "the source of this disclosure was management, specifically my Team Leader",
        "the dissemination of the allegation did not arise from informal speculation or misunderstanding among colleagues"
      ],
      "excludes": []
    },
    "X5": {
      "name": "Adverse reputational/prospects impact flowing from disclosure/predetermination",
      "definition": "Negative consequences on reputation or prospects due to the disclosure or predetermination.",
      "positive_indicators": [
        "the disclosure of the reason for my dismissal had material and adverse consequences",
        "the manner in which it was presented as an established finding, had damaged my professional reputation and prospects"
      ],
      "excludes": []
    }
  },
  "classes": {
    "DIRECT_X": {
      "definition": "Supports at least one of X1, X2, X3, or X4 (misconduct/behaviour undermining appeal integrity or confidentiality). X5 alone is not enough for DIRECT_X.",
      "min_support": 1,
      "evidence_required": True
    },
    "CONTRASTIVE": {
      "definition": "Describes proper appeal/open-minded review/independent reconsideration (useful comparison), but no misconduct matching X1..X4.",
      "evidence_required": True
    },
    "REMEDY": {
      "definition": "Primarily about compensation/mitigation/Polkey/Johnson v Unisys/Triggs/timelines/causation of loss and NOT about X1..X4.",
      "evidence_required": True
    },
    "IRRELEVANT": {
      "definition": "None of the above."
    }
  },
  "hard_rules": [
    "DIRECT_X requires supports_X contains at least one of [X1,X2,X3,X4] AND evidence_required=true.",
    "If only remedy topics appear (compensation/mitigation/timeline) and none of X1..X4 -> REMEDY.",
    "CONTRASTIVE must mention proper appeal/open-minded/independent review language and exclude misconduct.",
    "If evidence is missing or vague, prefer IRRELEVANT over DIRECT_X."
  ]
}


# =========================
# UTIL: Freeze Y to disk (non-negotiable)
# =========================
def freeze_y_to_disk(y: Dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        # Don't overwrite silently: confirm it's identical
        existing = json.loads(path.read_text(encoding="utf-8"))
        if existing != y:
            raise RuntimeError(f"Refusing to overwrite {path} with different content. Y must be frozen.")
        return
    path.write_text(json.dumps(y, indent=2, ensure_ascii=False), encoding="utf-8")


# =========================
# INPUT ITERATION (streaming, multi-appeal safe)
# =========================
def iter_jsonl_records(path: Path) -> Iterator[Dict[str, Any]]:
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            yield json.loads(line)

def iter_appeals_from_record(rec: Dict[str, Any]) -> Iterator[Tuple[int, Dict[str, Any]]]:
    """
    Many of your lines contain multiple appeals. Handle:
    - rec["appeals"] as list
    - else treat rec itself as one 'appeal-like' object
    """
    appeals = rec.get("appeals")
    if isinstance(appeals, list) and appeals:
        for i, a in enumerate(appeals):
            if isinstance(a, dict):
                yield i, a
            else:
                yield i, {"raw": a}
    else:
        yield 0, rec

def compute_item_id(src_file: Path, line_obj: Dict[str, Any], appeal_idx: int, appeal_obj: Dict[str, Any]) -> str:
    """
    Stable ID for resume cache. Prefer explicit identifiers; fallback to hash.
    """
    # Try common keys if they exist
    preferred = (
        appeal_obj.get("uk_eat_no")
        or appeal_obj.get("neutral_citation")
        or appeal_obj.get("case_id")
        or appeal_obj.get("filename")
        or line_obj.get("filename")
        or line_obj.get("source_file")
    )
    base = f"{src_file.name}::idx={appeal_idx}::pref={preferred or ''}"
    # Add small content fingerprint to avoid collisions when 'preferred' missing
    seed = (appeal_obj.get("reasoning_for_index") or appeal_obj.get("summary") or "")[:3000]
    h = hashlib.sha1((base + "||" + seed).encode("utf-8")).hexdigest()[:16]
    return f"{src_file.name}::{appeal_idx}::{preferred or 'NA'}::{h}"


# =========================
# PASS-1: input field selection
# =========================
def pick_record_text(appeal_obj: Dict[str, Any]) -> Tuple[str, str]:
    if appeal_obj.get("reasoning_for_index"):
        return appeal_obj["reasoning_for_index"], "reasoning_for_index"
    if appeal_obj.get("summary"):
        return appeal_obj["summary"], "summary"
    return "", "missing"


# =========================
# OLLAMA CALL (strict JSON output)
# =========================
def call_ollama_json(prompt: str, model: str = MODEL_NAME) -> Dict[str, Any]:
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.0,
            "num_predict": 350,
        }
    }
    r = requests.post(OLLAMA_URL, json=payload, timeout=120)
    r.raise_for_status()
    data = r.json()
    txt = data.get("response", "").strip()

    # Hard parse: try direct JSON, else attempt to extract first JSON object.
    try:
        return json.loads(txt)
    except Exception:
        # Extract first {...} block (best-effort) then parse
        start = txt.find("{")
        end = txt.rfind("}")
        if start != -1 and end != -1 and end > start:
            return json.loads(txt[start:end+1])
        raise RuntimeError(f"Model did not return valid JSON. Raw:\n{txt}")


# =========================
# PROMPT (Pass-1)
# =========================
def build_pass1_prompt(record_text: str, y: Dict[str, Any]) -> str:
    # Keep it compact but strict. Evidence snippet must be verbatim substring.
    y_compact = json.dumps(y, ensure_ascii=False)
    return (
        "You are a legal classifier. Return STRICT JSON ONLY, no extra text.\n\n"
        "TASK:\n"
        "Classify the record_text according to Y_v2 rules, and extract ONE evidence_snippet that is a VERBATIM substring of record_text.\n\n"
        "OUTPUT SCHEMA (STRICT):\n"
        '{ "classification": "DIRECT_X|CONTRASTIVE|REMEDY|IRRELEVANT",\n'
        '  "supports_X": ["X1","X2","X3","X4","X5"],\n'
        '  "evidence_snippet": "verbatim snippet from record_text",\n'
        '  "confidence": 0-100,\n'
        '  "note": "ONE sentence"\n'
        "}\n\n"
        "Y_V2:\n"
        f"{y_compact}\n\n"
        "record_text:\n"
        f"{record_text}\n"
    )


# =========================
# HARD VALIDATION LAYER
# =========================
REMEDY_SIGNALS = [
    "polkey", "mitigation", "compensation", "loss", "causation", "timeline",
    "johnson v unisys", "johnson v", "triggs", "remedy", "remedies", "uplift",
    "acaser", "injury to feelings", "schedule of loss"
]

def norm(s: str) -> str:
    return " ".join((s or "").split())

def contains_remedy_only(record_text: str) -> bool:
    t = (record_text or "").lower()
    hit = any(sig in t for sig in REMEDY_SIGNALS)
    # "only" here means: remedy signals present and no direct X1-X4 style triggers.
    # We don't have perfect triggers; we rely on supports_X check + snippet verbatim enforcement.
    return hit

def validate_and_fix(
    raw_out: Dict[str, Any],
    record_text: str
) -> Tuple[Dict[str, Any], List[str]]:
    reasons = []
    out = dict(raw_out or {})

    classification = (out.get("classification") or "").strip()
    supports = out.get("supports_X") or []
    if not isinstance(supports, list):
        supports = []
    supports = [str(x).strip() for x in supports if str(x).strip()]
    snippet = out.get("evidence_snippet") or ""
    snippet_n = norm(snippet)
    text_n = norm(record_text)

    # Rule B: snippet must be non-empty and verbatim found in record_text
    if not snippet_n or snippet_n not in text_n:
        out["classification"] = "IRRELEVANT"
        out["supports_X"] = []
        reasons.append("RuleB: evidence_snippet empty or not verbatim found -> IRRELEVANT")
        return out, reasons

    # Rule A: DIRECT_X must include X1..X4
    if classification == "DIRECT_X":
        if not any(x in supports for x in ("X1", "X2", "X3", "X4")):
            # downgrade: if remedy-ish -> REMEDY else IRRELEVANT
            if contains_remedy_only(record_text):
                out["classification"] = "REMEDY"
                reasons.append("RuleA: DIRECT_X without X1..X4 -> REMEDY (remedy signals detected)")
            else:
                out["classification"] = "IRRELEVANT"
                reasons.append("RuleA: DIRECT_X without X1..X4 -> IRRELEVANT")
            out["supports_X"] = [x for x in supports if x in ("X5",)]  # keep X5 only if present
            return out, reasons

    # Rule C: remedy-only + no X1..X4 -> force REMEDY (unless already IRRELEVANT)
    if classification in ("CONTRASTIVE", "DIRECT_X", "REMEDY"):
        if contains_remedy_only(record_text) and not any(x in supports for x in ("X1","X2","X3","X4")):
            out["classification"] = "REMEDY"
            reasons.append("RuleC: remedy-topic signals + no X1..X4 -> REMEDY")

    # Basic schema cleanup
    if out.get("classification") not in ("DIRECT_X", "CONTRASTIVE", "REMEDY", "IRRELEVANT"):
        out["classification"] = "IRRELEVANT"
        out["supports_X"] = []
        reasons.append("Invalid classification value -> IRRELEVANT")

    # Confidence clamp
    try:
        c = int(out.get("confidence"))
    except Exception:
        c = 0
    out["confidence"] = max(0, min(100, c))

    out["supports_X"] = [x for x in supports if x in ("X1","X2","X3","X4","X5")]

    note = out.get("note") or ""
    out["note"] = note.strip()[:300]  # hard cap

    out["evidence_snippet"] = snippet.strip()
    return out, reasons


# =========================
# RESUME CACHE
# =========================
def load_processed_ids(pass1_out: Path) -> set:
    done = set()
    if not pass1_out.exists():
        return done
    with pass1_out.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                item_id = obj.get("item_id")
                if item_id:
                    done.add(item_id)
            except Exception:
                continue
    return done


# =========================
# MAIN SCAN
# =========================
def run_pass1():
    freeze_y_to_disk(Y_v2, Y_JSON_PATH)
    processed = load_processed_ids(PASS1_OUT)
    print(f"Y frozen at: {Y_JSON_PATH}")
    print(f"Existing pass1 lines: {len(processed)} (resume cache)")

    n_written = 0
    PASS1_OUT.parent.mkdir(parents=True, exist_ok=True)

    with PASS1_OUT.open("a", encoding="utf-8") as out_f:
        for src in INPUT_FILES:
            if not src.exists():
                print(f"WARNING missing: {src}")
                continue

            for line_obj in iter_jsonl_records(src):
                for appeal_idx, appeal_obj in iter_appeals_from_record(line_obj):
                    item_id = compute_item_id(src, line_obj, appeal_idx, appeal_obj)
                    if item_id in processed:
                        continue

                    record_text, source_field = pick_record_text(appeal_obj)

                    # If missing record_text, still write an IRRELEVANT result deterministically.
                    if not record_text.strip():
                        result = {
                            "item_id": item_id,
                            "source_file": src.name,
                            "appeal_index": appeal_idx,
                            "source_field": source_field,
                            "classification": "IRRELEVANT",
                            "supports_X": [],
                            "evidence_snippet": "",
                            "confidence": 0,
                            "note": "Missing record_text.",
                            "hard_rule_actions": ["Missing record_text -> IRRELEVANT"],
                        }
                        out_f.write(json.dumps(result, ensure_ascii=False) + "\n")
                        out_f.flush()
                        processed.add(item_id)
                        n_written += 1
                        if DEBUG_MAX and n_written >= DEBUG_MAX:
                            print("DEBUG_MAX reached.")
                            return
                        continue

                    prompt = build_pass1_prompt(record_text, Y_v2)
                    raw = call_ollama_json(prompt, model=MODEL_NAME)
                    fixed, actions = validate_and_fix(raw, record_text)

                    # Persist minimal but useful diagnostics
                    result = {
                    # ---- identity / provenance ----
                    "item_id": item_id,
                    "source_file": src.name,
                    "appeal_index": appeal_idx,

                    "filename": (
                        appeal_obj.get("filename")
                        or line_obj.get("filename")
                    ),
                    "neutral_citation": (
                        appeal_obj.get("neutral_citation")
                        or line_obj.get("neutral_citation")
                    ),
                    "uk_eat_no": (
                        appeal_obj.get("uk_eat_no")
                        or line_obj.get("uk_eat_no")
                    ),

                    "doc_type": appeal_obj.get("doc_type"),
                    "appeal_type": appeal_obj.get("appeal_type"),
                    "who_appealed": appeal_obj.get("who_appealed"),
                    "outcome": appeal_obj.get("outcome"),
                    "successful": appeal_obj.get("successful"),
                    "favourable_to": appeal_obj.get("favourable_to"),

                    # ---- Pass-1 mechanics ----
                    "source_field": source_field,
                    "classification": fixed["classification"],
                    "supports_X": fixed["supports_X"],
                    "evidence_snippet": fixed["evidence_snippet"],
                    "confidence": fixed["confidence"],
                    "note": fixed["note"],
                    "hard_rule_actions": actions,
                }

                    out_f.write(json.dumps(result, ensure_ascii=False) + "\n")
                    out_f.flush()
                    processed.add(item_id)
                    n_written += 1

                    if n_written % 50 == 0:
                        print(f"Wrote {n_written} new items...")

                    if DEBUG_MAX and n_written >= DEBUG_MAX:
                        print("DEBUG_MAX reached.")
                        return

    print(f"Done. New items written: {n_written}")


# =========================
# SUMMARY STATS
# =========================
def summarize_pass1(pass1_out: Path = PASS1_OUT):
    if not pass1_out.exists():
        print("No pass1_results.jsonl found.")
        return

    counts = {"DIRECT_X":0,"CONTRASTIVE":0,"REMEDY":0,"IRRELEVANT":0}
    x_freq = {"X1":0,"X2":0,"X3":0,"X4":0,"X5":0}
    fallback_summary = 0
    direct_top = []  # (confidence, item)

    with pass1_out.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            cls = obj.get("classification", "IRRELEVANT")
            if cls not in counts:
                cls = "IRRELEVANT"
            counts[cls] += 1

            if obj.get("source_field") == "summary":
                fallback_summary += 1

            for x in obj.get("supports_X") or []:
                if x in x_freq:
                    x_freq[x] += 1

            if cls == "DIRECT_X":
                direct_top.append((int(obj.get("confidence", 0)), obj))

    direct_top.sort(key=lambda t: t[0], reverse=True)
    top50 = direct_top[:50]

    print("=== counts by classification ===")
    for k,v in counts.items():
        print(f"{k:12s}: {v}")

    print("\n=== supports_X frequency ===")
    for k,v in x_freq.items():
        print(f"{k}: {v}")

    print(f"\n=== used fallback summary ===\nsummary_used: {fallback_summary}")

    print("\n=== top 50 DIRECT_X by confidence ===")
    for c, obj in top50:
        print(f"- {c:3d} | {obj.get('uk_eat_no') or obj.get('neutral_citation') or obj.get('item_id')}")


# =========================
# RUN
# =========================
run_pass1()
summarize_pass1()


In [12]:
import json
import pandas as pd
from pathlib import Path

PASS1_PATH = Path("/home/hello/Projects/Statements/output/pass1_results.jsonl")

rows = []
with PASS1_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

df = pd.DataFrame(rows)

def filename_from_item_id(item_id: str):
    if not isinstance(item_id, str):
        return None
    parts = item_id.split("::")
    return parts[2] if len(parts) >= 3 else None  # this is the pdf token in your format

df["filename_guess"] = df["item_id"].apply(filename_from_item_id)

direct_df = df[df["classification"] == "DIRECT_X"].sort_values("confidence", ascending=False)

print(f"Loaded {len(df)} rows")
direct_df[[
    "filename_guess",
    "item_id",
    "confidence",
    "supports_X",
    "evidence_snippet",
    "note"
]]


Loaded 1715 rows


,filename_guess,item_id,confidence,supports_X,evidence_snippet,note
175,Northbay_Pelagic_Ltd_v_Mr_Colin__Anderson__UKE...,judgments_claimant_fav_faiss_chunk_with_reason...,90,[X1],the disciplinary process was tainted by potent...,Evidence of pre-determination supports X1.
